# Shulchan Aruch → JSONL Pipeline v2.1
## Chunking סמנטי + חילוץ מקורות + הפרדת מחבר/רמ״א + סיווג ר״ת 6-כיווני

### שדות content בכל chunk:

| שדה | מטרה | טיפול |
|---|---|---|
| `text_for_embedding` | Vector search | `''` → `״` (נרמול) |
| `text_for_embedding_expanded` | A/B test | ר״ת אמיתיים → הרחבה, שאר → `״` |
| `text_for_bm25` | BM25 keyword search | טקסט גולמי ללא שינוי |
| `text_for_llm` | הזנה ל-LLM | ר״ת מורחבים + תגי `[הגהת הרמ״א]` |

### הוראות הרצה:
1. הרץ תאים 1–6 (הגדרות + פונקציות)
2. הרץ **תא 7** — הרצה מלאה + שמירה + הורדה

הקבצים ירדו אוטומטית בסוף תא 7.


In [1]:
# ============================================================
# תא 1 — ייבוא + הגדרות + זיהוי קובץ קלט
# ============================================================
import json
import re
import os
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# === CONFIG ===
CHELEK = "אורח חיים"
MAX_CHUNK_WORDS = 350
TARGET_CHUNK_WORDS = 200
MIN_CHUNK_WORDS = 25
OVERLAP_CHARS = 150

# === נתיב פלט ===
if IN_COLAB:
    OUTPUT_DIR = "/content/pipeline_output"
else:
    OUTPUT_DIR = os.path.join(os.getcwd(), "pipeline_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === זיהוי קובץ קלט ===
INPUT_FILE = None
for p in sorted(Path('.').glob('*.txt')):
    INPUT_FILE = str(p)
    break
if INPUT_FILE is None and IN_COLAB:
    print("לא נמצא קובץ TXT. העלה קובץ:")
    uploaded = colab_files.upload()
    for name in uploaded:
        if name.endswith('.txt'):
            INPUT_FILE = name
            break
if INPUT_FILE is None:
    raise FileNotFoundError("לא נמצא קובץ TXT.")

print(f"קלט: {os.path.abspath(INPUT_FILE)}")
print(f"גודל: {os.path.getsize(INPUT_FILE):,} bytes")
print(f"פלט: {os.path.abspath(OUTPUT_DIR)}/")


לא נמצא קובץ TXT. העלה קובץ:


Saving Shulchan Aruch Orach Chaim Part 1.txt to Shulchan Aruch Orach Chaim Part 1.txt
קלט: /content/Shulchan Aruch Orach Chaim Part 1.txt
גודל: 298,200 bytes
פלט: /content/pipeline_output/


In [2]:
# ============================================================
# תא 2 — גימטריה + סיווג ר"ת 6-כיווני + חילוץ מקורות
# ============================================================

GEMATRIA_MAP = {
    'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,
    'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90
}

def hebrew_to_gematria(hebrew_str: str) -> int:
    cleaned = re.sub(r'["׳״\'\' \s]', '', hebrew_str or '')
    return sum(GEMATRIA_MAP.get(c, 0) for c in cleaned)


# ============================================================
# ACRONYM CLASSIFIER v1.1
# מסווג 599 מופעי '' ב-6 קטגוריות. רק 22 (4%) מורחבים.
# ============================================================

NUMBER_KW_BEFORE = re.compile(
    r'(?:סימן|סעיף|דף|פרק|כלל|חלק|נתיב|שורש|הלכה|מהלכות|ובו|ריש|סוף)\s*$'
)
NUMBER_KW_AFTER = re.compile(
    r'^\s*(?:סעיפים|סעיפי|גודלים|טפחים|חוטים|פעמים|אמות|תפירות|אצבעות|תיבות|ימים|שנים|שנות|חדש)'
)
AMUD_SET = {"ע''ב", "ע''א"}

POSEK_ROOTS = {
    "רא''ש","רש''י","רמב''ם","רמב''ן","רשב''א","ר''ן","סמ''ג","סמ''ק",
    "מהרי''ל","מהרי''א","מהרי''ק","מהרי''ו","מהרי''ח",
    "מהר''ם","מהר''י","מהר''א","מהרא''י","ריב''ש","ר''י",
    "הרא''ש","הר''י","הר''ן","הר''ר","הר''מ",
    "הרשב''א","הרמב''ם","הרמב''ן","הרי''ף","הראב''ד",
    "הרד''ק","הסמ''ג","הסמ''ק","תשב''ץ",
}

LETTER_ROOTS = {
    "אל''ף","שי''ן","דלי''ת","נו''ן","צד''י","עי''ן","עיי''ן",
    "למ''ד","מ''ם","ה''א","כ''ף","בי''ת","יו''ד","יוד''י",
    "חי''ת","קו''ף","רי''ש","פ''א","וי''ו","זיי''ן","זייני''ן",
    "היו''ד","הנו''ן","החי''ת","הקו''ף","הלמ''ד","הדלי''ת",
    "האל''ף","השי''ן","המ''ם","הה''א","הכ''ף","העי''ן",
    "יודי''ן","אלפי''ן","שיני''ן","עייני''ן","התוי''ן","ווי''ן",
    "צדי''ק","שד''י","ג''ץ",
}

FOREIGN_WORDS = {
    "בלע''ז","קאפ''ה","קאפיל''ה","טאסק''ה","רייטי''ר",
    "קראבאנ''א","גוחא''ש","מינטיג''י","דואלמני''ש",
    "קאפטאני''ש","פידיני''ש","אשטרינג''ה","שטרינג''ה",
    "פאדו''ה","אליאדור''ש","טאליאדור''ש",
    "בוק''א","הקובד''ו","לקובד''ו","שי''ד",
}

EXPAND_DICT = {
    "הקב''ה": "הקדוש ברוך הוא", "שהקב''ה": "שהקדוש ברוך הוא",
    "להקב''ה": "להקדוש ברוך הוא",
    "השי''ת": "השם יתברך", "הוי''ה": "הויה", "אדנ''י": "אדני",
    "שעטנ''ז": "שעטנז", "אע''פ": "אף על פי",
}

HEB_PREFIXES = [
    'ולה','וכה','שכה','שלה','וכל',
    'שב','וב','וה','ול','של','לה','מה','שה','כה','וכ','שכ','לכ','בכ','וד','בד','דה',
    'ו','ה','ב','כ','ל','מ','ש','ד',
]

DOUBLE_GERESH_RE = re.compile(r"([א-ת]{1,7})''([א-ת]{1,7})")

def _match_with_prefix(token, root_set):
    if token in root_set:
        return True
    for pfx in HEB_PREFIXES:
        if token.startswith(pfx) and len(token) > len(pfx) + 2:
            if token[len(pfx):] in root_set:
                return True
    return False

def classify_double_geresh(token, left_ctx='', right_ctx=''):
    if "''" not in token:
        return 'other'
    if token in EXPAND_DICT:
        return 'abbreviation'
    if token in AMUD_SET:
        return 'numbering_ref'
    if NUMBER_KW_BEFORE.search(left_ctx):
        return 'numbering_ref'
    if _match_with_prefix(token, POSEK_ROOTS):
        return 'posek'
    if _match_with_prefix(token, LETTER_ROOTS):
        return 'letter_name'
    if token in FOREIGN_WORDS or _match_with_prefix(token, FOREIGN_WORDS):
        return 'foreign'
    if NUMBER_KW_AFTER.match(right_ctx):
        return 'standalone_number'
    parts = token.split("''")
    if len(parts) == 2 and len(parts[0]) <= 2 and len(parts[1]) == 1:
        return 'standalone_number'
    return 'abbreviation'

def process_acronyms(text, mode='normalize'):
    """mode: normalize ('' -> ״), expand (ר"ת -> הרחבה), keep (ללא שינוי)"""
    if not text or mode == 'keep':
        return text
    if mode == 'normalize':
        result = text.replace("''", "\u05F4")
        result = result.replace("וכו'", "וכו\u05F3")
        result = result.replace("וגו'", "וגו\u05F3")
        return result
    if mode == 'expand':
        def replacer(m):
            full = m.group(0)
            left = text[max(0, m.start()-30):m.start()]
            right = text[m.end():min(len(text), m.end()+20)]
            cat = classify_double_geresh(full, left, right)
            if cat == 'abbreviation' and full in EXPAND_DICT:
                return EXPAND_DICT[full]
            return full.replace("''", "\u05F4")
        result = DOUBLE_GERESH_RE.sub(replacer, text)
        result = result.replace("וכו'", "וכולי")
        result = result.replace("וגו'", "וגומר")
        return result
    return text


# ============================================================
# חילוץ מקורות מסוגריים
# ============================================================
SOURCE_KEYWORDS = [
    'טור', 'רא"ש', 'רשב"א', 'רמב"ם', 'רמב"ן', 'רי"ף', 'ר"ן', 'רש"י',
    'תוספות', 'מרדכי', 'סמ"ג', 'סמ"ק', 'רוקח', 'אור זרוע', 'בית יוסף',
    'כל בו', 'אבודרהם', 'מיימוני', 'מהרי"ל', 'תרומת הדשן', 'תשובת',
    'תשובות', 'הגהות', 'שבולי', 'ברוך שאמר', 'אורחות חיים', 'העיטור',
    'מורה נבוכים', 'דרך ארץ', 'אגור', 'מהר"ם'
]
PAREN_RE = re.compile(r'\(([^()]+)\)')

def extract_sources_and_clean(text: str):
    sources = []
    cross_references = []
    def replacer(match):
        content = match.group(1).strip()
        if content.startswith('פירוש'):
            return match.group(0)
        if any(kw in content for kw in ['עיין', 'לקמן', 'לעיל']):
            cross_references.append(content)
            return ''
        if any(kw in content for kw in SOURCE_KEYWORDS) or ('דף' in content) or ('פרק' in content):
            sources.append(content)
            return ''
        return match.group(0)
    clean_text = text or ''
    prev = None
    while prev != clean_text:
        prev = clean_text
        clean_text = PAREN_RE.sub(replacer, clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    clean_text = clean_text.replace(' .', '.').replace(' :', ':').replace(',.', '.')
    return clean_text, sources, cross_references



# ============================================================
# הסרת ביטויי הפניה צולבת מחוץ לסוגריים
# 27 ביטויי "ראה שם" שאינם תוכן הלכתי (למשל: "כדלקמן סימן ק"ח")
# מוסרים מ-embedding בלבד. נשמרים ב-bm25 ו-llm.
# ============================================================

_CROSS_REF_PATTERNS = [
    re.compile(r'(?:ו)?(?:כמו\s+ש)?(?:י|נ|ש)תבאר\s+(?:ב(?:טור|סימן|יורה|לקמן|לעיל))\S*(?:\s+(?:יורה דעה|אורח חיים|חושן משפט))?(?:\s+סימן\s+\S+)?(?:\s+סעיף\s+\S+)?[.:]?'),
    re.compile(r'(?:ו)?כמבואר\s+ב\S+(?:\s+(?:יורה דעה|אורח חיים))?(?:\s+ובשאר\s+פוסקים)?'),
    re.compile(r'(?:ו)?כמו\s+שכתבתי\s+(?:לעיל|לקמן)\s+סימן\s+\S+[,.]?'),
    re.compile(r'(?:ו)?כד(?:לקמן|לעיל)\s+סימן\s+\S+(?:\s+סעיף\s+\S+)?[.:]?'),
    re.compile(r'(?:ו)?עיין\s+(?:לקמן|לעיל)(?:\s+(?:ריש\s+|סוף\s+)?סימן\s+\S+)?(?:\s+סעיף\s+\S+)?[.:]?'),
    re.compile(r'(?:ו)?עיין\s+ב(?:יורה דעה|אורח חיים|חושן משפט)\s+סימן\s+\S+'),
    re.compile(r'וכן\s+הוא\s+(?:לקמן|לעיל)\s+סימן\s+\S+[.:]?'),
]

def remove_cross_ref_phrases(text):
    """מסיר ביטויי הפניה צולבת (כדלקמן סימן X, ועיין לעיל...) מהטקסט.
    מיועד ל-text_for_embedding בלבד."""
    if not text:
        return text
    for pat in _CROSS_REF_PATTERNS:
        text = pat.sub('', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text



# ============================================================
# תיוג שמות פוסקים וספרים
# [פוסק:רמב״ם]  [פוסק:רבינו תם]  [ספר:טור]
# ============================================================

_POSEK_TAG_MAP = {
    'רמב״ם': 'רמב״ם', 'רא״ש': 'רא״ש', 'רש״י': 'רש״י',
    'רשב״א': 'רשב״א', 'רמב״ן': 'רמב״ן', 'ר״ן': 'ר״ן', 'ר״י': 'ר״י',
    'סמ״ג': 'סמ״ג', 'סמ״ק': 'סמ״ק', 'רי״ף': 'רי״ף',
    'מהרי״ל': 'מהרי״ל', 'מהרי״א': 'מהרי״א', 'מהרי״ק': 'מהרי״ק',
    'מהרי״ו': 'מהרי״ו', 'מהרי״ח': 'מהרי״ח',
    'מהר״ם': 'מהר״ם', 'מהר״י': 'מהר״י', 'מהר״א': 'מהר״א',
    'מהרא״י': 'מהרא״י', 'ריב״ש': 'ריב״ש',
    'הרמב״ם': 'רמב״ם', 'הרא״ש': 'רא״ש', 'הרשב״א': 'רשב״א',
    'הרמב״ן': 'רמב״ן', 'הרי״ף': 'רי״ף', 'הראב״ד': 'ראב״ד',
    'הרד״ק': 'רד״ק', 'הסמ״ג': 'סמ״ג', 'הסמ״ק': 'סמ״ק',
    'הר״י': 'ר״י', 'הר״ן': 'ר״ן', 'הר״ר': 'ר״ר', 'הר״מ': 'ר״מ',
    'תשב״ץ': 'תשב״ץ',
}

_TAG_PREFIXES = ['ולה','וכה','של','לה','מה','כה','וכ',
                 'וב','וה','ול','ו','ה','ב','כ','ל','מ','ש']

_all_posek_forms = []
for _r in sorted(_POSEK_TAG_MAP.keys(), key=len, reverse=True):
    _all_posek_forms.append(re.escape(_r))
    for _p in _TAG_PREFIXES:
        _all_posek_forms.append(re.escape(_p + _r))
_all_posek_forms.sort(key=len, reverse=True)
_POSEK_RE = re.compile('(' + '|'.join(_all_posek_forms) + ')' + r'(?=[^א-ת\u05F4]|$)')

_PERSON_NAMES = {
    'רבינו תם': 'רבינו תם',
    'רבינו ירוחם': 'רבינו ירוחם',
    'רבינו יונה': 'רבינו יונה',
}
_person_pats = []
for _n in sorted(_PERSON_NAMES.keys(), key=len, reverse=True):
    _person_pats.append(r'(?:ו?ל|ו?כ|של|ו)?' + re.escape(_n))
_person_pats.sort(key=len, reverse=True)
_PERSON_RE = re.compile('(' + '|'.join(_person_pats) + ')' + r'(?=[^א-ת]|$)')

_SEFER_RE = re.compile(r'(?<![פא-ת])(?:ב|כ|וה|שב|של)(טור)(?=[\s.:,)]|$)')

def tag_references(text):
    """מתייג שמות פוסקים וספרים: [פוסק:X] [ספר:X]"""
    if not text:
        return text

    # 1) פוסקים עם ״
    def _tag_posek(m):
        token = m.group(1)
        for pfx in _TAG_PREFIXES:
            if token.startswith(pfx) and token[len(pfx):] in _POSEK_TAG_MAP:
                return pfx + '[פוסק:' + _POSEK_TAG_MAP[token[len(pfx):]] + ']'
        if token in _POSEK_TAG_MAP:
            return '[פוסק:' + _POSEK_TAG_MAP[token] + ']'
        return token
    text = _POSEK_RE.sub(_tag_posek, text)

    # 2) רבינו X
    def _tag_person(m):
        token = m.group(1)
        for name, display in _PERSON_NAMES.items():
            if token.endswith(name):
                prefix = token[:-len(name)]
                return prefix + '[פוסק:' + display + ']'
        return token
    text = _PERSON_RE.sub(_tag_person, text)

    # 3) ספר טור
    text = _SEFER_RE.sub(lambda m: m.group(0).replace('טור', '[ספר:טור]'), text)

    return text


print("✅ תא 2")


✅ תא 2


In [3]:
# ============================================================
# תא 3 — זיהוי מחלוקת, פוסקים, נושאים, הפניות
# ============================================================

MACHLOKET_INDICATORS = [
    'ויש אומרים', 'יש אומרים', 'ויש מי שאומר', 'יש מי שאומר',
    'ויש חולקים', 'יש חולקים', 'ויש מתירים', 'ויש אוסרים',
    'יש להחמיר', 'יש להקל', 'ויש מקילין', 'פלוגתא', 'מחלוקת',
]
def detect_machloket(text): return any(i in text for i in MACHLOKET_INDICATORS)

def detect_psak_type(text):
    t = []
    if 'לכתחילה' in text or 'לכתחלה' in text: t.append('לכתחילה')
    if 'בדיעבד' in text or 'דיעבד' in text: t.append('בדיעבד')
    if any(w in text for w in ['מנהג', 'נוהגים', 'נוהגין', 'נהגו']): t.append('מנהג')
    return ','.join(t) if t else 'מדינא'

KNOWN_POSKIM = [
    'רמב"ם', 'רש"י', 'רא"ש', 'טור', 'בית יוסף', 'מרדכי',
    'סמ"ג', 'סמ"ק', 'רשב"א', 'אגור', 'רוקח', 'אבודרהם',
    'ברוך שאמר', 'כל בו', 'אורחות חיים', 'העיטור',
    'תרומת הדשן', 'מהר"ם', 'מהרי"ל', 'אור זרוע', 'תוספות',
]
def extract_poskim(text): return [p for p in KNOWN_POSKIM if p in text]

CROSS_REF_RE = re.compile(r'(?:ועיין\s+)?(?:לקמן|לעיל)\s+(?:סימן\s+)?([א-ת]{1,3})')
def parse_cross_refs(text):
    return [{'hebrew': m.group(1), 'number': hebrew_to_gematria(m.group(1))}
            for m in CROSS_REF_RE.finditer(text)]

TOPIC_KEYWORDS = {
    'תפילין': ['תפילין', 'בתים', 'רצועות'],
    'ציצית': ['ציצית', 'טלית', 'כנף', 'גדיל'],
    'קריאת שמע': ['קריאת שמע', 'שמע ישראל'],
    'תפלה': ['תפלה', 'שמונה עשרה', 'שליח ציבור'],
    'ברכות': ['ברכה', 'ברכות', 'אמן'],
    'סת"ם': ['כתיבה', 'קולמוס', 'דיו', 'קלף', 'אותיות'],
    'נטילת ידים': ['נטילת ידים', 'רחיצה'],
    'בית הכסא': ['בית הכסא', 'צואה'],
    'הנהגת הבוקר': ['השכמה', 'בוקר', 'לבישת בגדים'],
}
def detect_topics(title, text, group):
    combined = title + ' ' + text + ' ' + group
    t = [tp for tp, kws in TOPIC_KEYWORDS.items() if any(kw in combined for kw in kws)]
    return t if t else ['כללי']


print("✅ תא 3")


✅ תא 3


In [4]:
# ============================================================
# תא 4 — Parser: סימן/סעיף + הפרדת מחבר/רמ"א
# ============================================================

SIMAN_RE = re.compile(r'^סימן\s+([א-ת][א-ת"׳״\']*)\s*-\s*(.*)$')
SEIF_RE  = re.compile(r'^([א-ת]{1,3})\.\s+')
SEPARATOR_RE = re.compile(r'^_{5,}$')

@dataclass
class ParsedSeif:
    siman_num: int; seif_num: int; seif_hebrew: str; mechaber_text: str
    rama_texts: list = field(default_factory=list)
    hilchot_group: str = ''; siman_title: str = ''

    @property
    def full_text(self):
        parts = [self.mechaber_text]
        for r in self.rama_texts: parts.append(f'[הגהת הרמ"א] {r}')
        return '\n'.join(parts)

    @property
    def full_text_raw(self):
        parts = [self.mechaber_text]
        for r in self.rama_texts: parts.append(f'הגה: {r}')
        return ' '.join(parts)

    @property
    def has_rama(self): return len(self.rama_texts) > 0
    @property
    def word_count(self): return len(self.full_text.split())


@dataclass
class ParsedSiman:
    number: int; hebrew: str; title: str; seifim_count_stated: str; hilchot_group: str
    seifim: list = field(default_factory=list)


def parse_shulchan_aruch(filepath: str):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    simanim, hilchot_groups, all_seifim = [], {}, []
    cur_h, cur_s, cur_se, in_haga, started = '', None, None, False, False

    for raw_line in lines:
        line = raw_line.strip()
        if SEPARATOR_RE.match(line): started = True; continue
        if not started:
            if line.startswith('הלכות ') and 'ובו' not in line:
                cur_h = line; hilchot_groups.setdefault(cur_h, [])
            continue
        if not line: continue

        if line.startswith('הלכות ') and not SIMAN_RE.match(line):
            cur_h = line; hilchot_groups.setdefault(cur_h, []); continue

        sm = SIMAN_RE.match(line)
        if sm:
            if cur_se:
                if cur_s: cur_s.seifim.append(cur_se)
                all_seifim.append(cur_se); cur_se = None
            sh = sm.group(1).strip(); sn = hebrew_to_gematria(sh)
            rt = sm.group(2).strip()
            title = re.sub(r'\.?\s*ובו.*', '', rt).strip().rstrip('.')
            cur_s = ParsedSiman(sn, sh, title, '', cur_h)
            simanim.append(cur_s)
            if cur_h in hilchot_groups and sn not in hilchot_groups[cur_h]:
                hilchot_groups[cur_h].append(sn)
            in_haga = False; continue

        sem = SEIF_RE.match(line)
        if sem and not line.startswith('הגה:') and not line.startswith('(הגה:'):
            if cur_se:
                if cur_s: cur_s.seifim.append(cur_se)
                all_seifim.append(cur_se)
            sh2 = sem.group(1)
            cur_se = ParsedSeif(
                cur_s.number if cur_s else 0, hebrew_to_gematria(sh2), sh2,
                line[sem.end():].strip(),
                hilchot_group=cur_h, siman_title=cur_s.title if cur_s else '')
            in_haga = False; continue

        is_haga, haga_text = False, ''
        if line.startswith('הגה:'): is_haga, haga_text = True, line[4:].strip()
        elif line.startswith('(הגה:'): is_haga, haga_text = True, line[5:].rstrip(')').strip()
        if is_haga:
            if cur_se: cur_se.rama_texts.append(haga_text); in_haga = True
            continue

        if cur_se:
            if in_haga and cur_se.rama_texts: cur_se.rama_texts[-1] += ' ' + line
            else: cur_se.mechaber_text += ' ' + line; in_haga = False

    if cur_se:
        if cur_s: cur_s.seifim.append(cur_se)
        all_seifim.append(cur_se)
    return simanim, hilchot_groups, all_seifim


print("✅ תא 4")


✅ תא 4


In [5]:
# ============================================================
# תא 5 — Chunker + בניית מסמכים (4 שדות content)
# ============================================================

def semantic_chunk_siman(siman, target=TARGET_CHUNK_WORDS,
                         mx=MAX_CHUNK_WORDS, mn=MIN_CHUNK_WORDS):
    if not siman.seifim: return []
    groups, buf, bw = [], [], 0
    for seif in siman.seifim:
        sw = seif.word_count
        if sw > mx:
            if buf: groups.append(buf); buf, bw = [], 0
            groups.append([seif]); continue
        if bw + sw > target and buf:
            groups.append(buf); buf, bw = [], 0
        buf.append(seif); bw += sw
    if buf:
        if bw < mn and groups: groups[-1].extend(buf)
        else: groups.append(buf)
    return groups


def build_chunk_doc(seif_group, siman, chunk_idx, total_chunks, prev_overlap=''):
    seif_nums = [s.seif_num for s in seif_group]
    has_rama = any(s.has_rama for s in seif_group)

    raw_parts    = [f'סעיף {s.seif_hebrew}: {s.full_text_raw}' for s in seif_group]
    tagged_parts = [f'סעיף {s.seif_hebrew}: {s.full_text}' for s in seif_group]

    raw_text    = '\n'.join(raw_parts)
    tagged_text = '\n'.join(tagged_parts)

    # ניקוי + חילוץ
    normalized = process_acronyms(raw_text, 'normalize')
    clean_text, sources, cross_refs_parens = extract_sources_and_clean(normalized)
    clean_expanded = process_acronyms(clean_text, 'expand')

    cross_refs_inline = parse_cross_refs(raw_text)
    all_cross_refs = cross_refs_parens + [r['hebrew'] for r in cross_refs_inline]

    poskim    = extract_poskim(raw_text)
    machloket = detect_machloket(raw_text)
    psak_type = detect_psak_type(raw_text)
    topics    = detect_topics(siman.title, raw_text, siman.hilchot_group)

    ctx = f'[{CHELEK} > {siman.hilchot_group} > סימן {siman.hebrew} - {siman.title}]'
    sr  = str(seif_nums[0]) if len(seif_nums) == 1 else f'{seif_nums[0]}-{seif_nums[-1]}'
    sf  = f'__{chr(ord("a") + chunk_idx)}' if total_chunks > 1 else ''
    doc_id = f'sa_oc_{siman.number:03d}_{sr}{sf}'

    path = f'שולחן ערוך, {CHELEK}'
    if siman.hilchot_group: path += f', {siman.hilchot_group}'
    path += f', סימן {siman.hebrew} ({siman.title})'
    if len(seif_nums) == 1: path += f', סעיף {seif_group[0].seif_hebrew}'
    else: path += f', סעיפים {seif_group[0].seif_hebrew}-{seif_group[-1].seif_hebrew}'

    # === 4 שדות CONTENT ===
    # 1) embedding: normalize + הסרת הפניות צולבות
    clean_for_emb = remove_cross_ref_phrases(clean_text)
    t_emb = f'{ctx}\n{clean_for_emb}'
    if prev_overlap:
        t_emb = f'{ctx}\n[...] {process_acronyms(prev_overlap, "normalize")}\n{clean_for_emb}'

    # 2) embedding_expanded: הרחבה + הסרת הפניות
    clean_exp_for_emb = tag_references(remove_cross_ref_phrases(clean_expanded))
    t_exp = f'{ctx}\n{clean_exp_for_emb}'
    if prev_overlap:
        t_exp = f'{ctx}\n[...] {process_acronyms(prev_overlap, "expand")}\n{clean_exp_for_emb}'

    # 3) bm25: טקסט גולמי ללא שינוי
    t_bm25 = f'{ctx}\n{raw_text}'

    # 4) llm: מורחב + תגי הגהה
    t_llm = f'{ctx}\nנושא: {siman.hilchot_group}\n{tag_references(process_acronyms(tagged_text, "expand"))}'

    return {
        'doc_id': doc_id,
        'metadata': {
            'work_title': 'Shulchan Aruch', 'author': 'Rabbi Yosef Karo',
            'sub_work': 'Orach Chayim',
            'general_topic': siman.hilchot_group, 'siman_topic': siman.title,
            'topics': topics, 'sources': sources,
            'cross_references': all_cross_refs,
            'poskim_mentioned': poskim,
            'has_rama': has_rama, 'machloket': machloket, 'psak_type': psak_type,
            'hierarchy': {
                'chelek': CHELEK, 'hilchot_group': siman.hilchot_group,
                'level_1_num': siman.number, 'level_1_hebrew': siman.hebrew,
                'level_2_nums': seif_nums,
                'chunk_index': chunk_idx + 1, 'total_chunks': total_chunks
            },
            'parent_siman_id': f'sa_oc_{siman.number:03d}',
            'human_readable_path': path,
            'word_count': len(clean_for_emb.split()),
        },
        'content': {
            'text_for_embedding': t_emb,
            'text_for_embedding_expanded': t_exp,
            'text_for_bm25': t_bm25,
            'text_for_llm': t_llm,
        }
    }


def build_hilchot_index_doc(group_name, siman_list):
    lines = [group_name]
    for s in siman_list:
        lines.append(f'סימן {s.hebrew} ({s.number}) - {s.title} ({len(s.seifim)} סעיפים)')
    text = '\n'.join(lines)
    ctx = f'[{CHELEK} > {group_name}]'
    return {
        'doc_id': f'sa_oc_index_{group_name.replace(" ", "_")}',
        'metadata': {
            'work_title': 'Shulchan Aruch', 'sub_work': 'Orach Chayim',
            'general_topic': group_name, 'doc_type': 'hilchot_index',
            'hierarchy': {'chelek': CHELEK, 'hilchot_group': group_name},
            'word_count': len(text.split()),
        },
        'content': {
            'text_for_embedding': f'{ctx}\n{text}',
            'text_for_embedding_expanded': f'{ctx}\n{text}',
            'text_for_bm25': f'{ctx}\n{text}',
            'text_for_llm': text,
        }
    }


def generate_all_chunks(simanim, hilchot_groups):
    all_docs = []
    gs = {}
    for s in simanim: gs.setdefault(s.hilchot_group or 'כללי', []).append(s)
    for gn, gl in gs.items():
        if gl: all_docs.append(build_hilchot_index_doc(gn, gl))
    for siman in simanim:
        chunks = semantic_chunk_siman(siman)
        total = len(chunks); prev = ''
        for i, group in enumerate(chunks):
            doc = build_chunk_doc(group, siman, i, total, prev)
            all_docs.append(doc)
            last = group[-1].mechaber_text
            if len(last) > OVERLAP_CHARS:
                ol = last[-OVERLAP_CHARS:]; sp = ol.find(' ')
                prev = ol[sp:].strip() if sp > 0 else ol
            else: prev = ''
    return all_docs


print("✅ תא 5")


✅ תא 5


In [6]:
# ============================================================
# תא 6 — Validation + HyDE + שמירה
# ============================================================

def validate_parsing(simanim, hilchot_groups, all_seifim):
    r = ['=' * 60, '  Validation - Pipeline v2.1', '=' * 60]
    nums = sorted(set(s.number for s in simanim))
    r.append(f'\nסימנים: {len(nums)} (טווח {min(nums)}-{max(nums)})')
    missing = set(range(min(nums), max(nums) + 1)) - set(nums)
    r.append(f'⚠️  חסרים: {sorted(missing)}' if missing else '✅ אין חסרים')
    r.append(f'סעיפים: {len(all_seifim)}')
    issues = []
    for siman in simanim:
        if not siman.seifim: issues.append(f'  סימן {siman.number}: 0 סעיפים')
        sns = [s.seif_num for s in siman.seifim]
        for j in range(1, len(sns)):
            if sns[j] != sns[j-1] + 1:
                issues.append(f'  סימן {siman.number}: קפיצה {sns[j-1]}->{sns[j]}')
    if issues: r.append(f'\n⚠️  {len(issues)} בעיות:'); r.extend(issues[:20])
    else: r.append('✅ רצף תקין')
    r.append(f'\nקבוצות: {len(hilchot_groups)}')
    for g in hilchot_groups:
        sc = len([s for s in simanim if s.hilchot_group == g])
        se = sum(len(s.seifim) for s in simanim if s.hilchot_group == g)
        r.append(f'  {g}: {sc} סימנים, {se} סעיפים')
    rama = sum(1 for s in all_seifim if s.has_rama)
    r.append(f'\nרמ"א: {rama}/{len(all_seifim)}')
    return '\n'.join(r)


def add_synthetic_questions(docs):
    for doc in docs:
        if doc['metadata'].get('doc_type') == 'hilchot_index': continue
        m = doc['metadata']; q = []
        if m.get('siman_topic'): q.append(f'מה הדין ב{m["siman_topic"]}?')
        if m.get('machloket'): q.append(f'מהי המחלוקת ב{m["siman_topic"]}?')
        if m.get('has_rama'): q.append(f'מה דעת הרמ"א ב{m["siman_topic"]}?')
        m['synthetic_questions'] = q
    return docs


def save_jsonl(docs, filename):
    p = os.path.join(OUTPUT_DIR, filename); n = 0
    with open(p, 'w', encoding='utf-8') as f:
        for d in docs:
            if (d.get('content') or {}).get('text_for_embedding', '').strip():
                f.write(json.dumps(d, ensure_ascii=False) + '\n'); n += 1
    return p, n

def save_json(docs, filename):
    p = os.path.join(OUTPUT_DIR, filename)
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(docs, f, ensure_ascii=False, indent=2)
    return p

def save_txt(text, filename):
    p = os.path.join(OUTPUT_DIR, filename)
    with open(p, 'w', encoding='utf-8') as f: f.write(text)
    return p


print("✅ תא 6")


✅ תא 6


In [7]:
# ============================================================
# תא 7 — הרצה מלאה → שמירה → הורדה
# ============================================================

print(f'📥 קלט: {os.path.abspath(INPUT_FILE)}')
print(f'📁 פלט: {os.path.abspath(OUTPUT_DIR)}/')
print()

# --- 1. Parse ---
print('🔄 [1/5] פרסור...')
simanim, hilchot_groups, all_seifim = parse_shulchan_aruch(INPUT_FILE)
print(f'   ✅ {len(simanim)} סימנים, {len(all_seifim)} סעיפים, {len(hilchot_groups)} קבוצות')

# --- 2. Validate ---
print('🔄 [2/5] Validation...')
report = validate_parsing(simanim, hilchot_groups, all_seifim)
save_txt(report, 'validation_report.txt')
print(report)

# --- 3. Chunk ---
print('\n🔄 [3/5] Chunking סמנטי...')
all_docs = generate_all_chunks(simanim, hilchot_groups)
content_docs = [d for d in all_docs if d['metadata'].get('doc_type') != 'hilchot_index']
print(f'   ✅ {len(all_docs)} chunks ({len(all_docs)-len(content_docs)} index + {len(content_docs)} content)')

# --- 4. HyDE ---
print('🔄 [4/5] שאלות סינתטיות...')
all_docs = add_synthetic_questions(all_docs)

# --- 5. Save ---
print('🔄 [5/5] שמירה...')
jsonl_path, jsonl_count = save_jsonl(all_docs, 'shulchan_aruch_v2.jsonl')
json_path = save_json(all_docs, 'shulchan_aruch_v2_full.json')

# === סטטיסטיקות ===
wc = [d['metadata']['word_count'] for d in content_docs]
total = len(content_docs)
print(f'\n{"="*60}')
print(f'  סטטיסטיקות')
print(f'{"="*60}')
print(f'  Chunks: {jsonl_count}')
print(f'  מילים: min={min(wc)}, max={max(wc)}, avg={sum(wc)//len(wc)}, median={sorted(wc)[len(wc)//2]}')
print(f'  רמ"א: {sum(1 for d in content_docs if d["metadata"].get("has_rama"))} ({100*sum(1 for d in content_docs if d["metadata"].get("has_rama"))//total}%)')
print(f'  מחלוקת: {sum(1 for d in content_docs if d["metadata"].get("machloket"))}')
print(f'  מקורות: {sum(1 for d in content_docs if d["metadata"].get("sources"))}')

# === בדיקת ר"ת ===
emb_apost = sum(d['content']['text_for_embedding'].count("''") for d in all_docs)
bm_apost = sum(d['content']['text_for_bm25'].count("''") for d in all_docs)
emb_gersh = sum(d['content']['text_for_embedding'].count('\u05F4') for d in all_docs)
print(f'\n  ר"ת: embedding \'\' ={emb_apost} ״={emb_gersh} | bm25 \'\' ={bm_apost}')

# === הדגמת expand ===
for d in content_docs:
    exp = d['content']['text_for_embedding_expanded']
    if 'הקדוש ברוך הוא' in exp:
        idx = exp.find('הקדוש ברוך הוא')
        print(f'\n  דוגמת expand: ...{exp[max(0,idx-15):idx+25]}...')
        break

# === קבצי פלט ===
print(f'\n{"="*60}')
print(f'  📂 קבצי פלט:')
print(f'{"="*60}')
output_files = []
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    fsize = os.path.getsize(fpath)
    output_files.append((fname, fpath, fsize))
    print(f'  📄 {fname}  ({fsize:,} bytes)')
print(f'\n  📂 {os.path.abspath(OUTPUT_DIR)}/')

# === הורדה ===
if IN_COLAB:
    print('\n⬇️  מוריד...')
    from google.colab import files as colab_files
    for fname, fpath, _ in output_files:
        colab_files.download(fpath)
        print(f'  ✅ {fname}')
    print('\n✅ כל הקבצים הורדו!')
else:
    print(f'\n✅ סיום.')


📥 קלט: /content/Shulchan Aruch Orach Chaim Part 1.txt
📁 פלט: /content/pipeline_output/

🔄 [1/5] פרסור...
   ✅ 126 סימנים, 751 סעיפים, 6 קבוצות
🔄 [2/5] Validation...
  Validation - Pipeline v2.1

סימנים: 126 (טווח 1-126)
✅ אין חסרים
סעיפים: 751
✅ רצף תקין

קבוצות: 6
  הלכות הנהגת אדם בבוקר: 7 סימנים, 64 סעיפים
  הלכות ציצית: 17 סימנים, 92 סעיפים
  הלכות תפילין: 21 סימנים, 152 סעיפים
  הלכות ברכת השחר ושאר ברכות: 12 סימנים, 94 סעיפים
  הלכות קריאת שמע: 31 סימנים, 149 סעיפים
  הלכות תפלה: 38 סימנים, 200 סעיפים

רמ"א: 228/751

🔄 [3/5] Chunking סמנטי...
   ✅ 220 chunks (6 index + 214 content)
🔄 [4/5] שאלות סינתטיות...
🔄 [5/5] שמירה...

  סטטיסטיקות
  Chunks: 220
  מילים: min=12, max=356, avg=129, median=144
  רמ"א: 161 (75%)
  מחלוקת: 76
  מקורות: 157

  ר"ת: embedding '' =0 ״=318 | bm25 '' =569

  דוגמת expand: ...לך מלכי המלכים הקדוש ברוך הוא אם לא מרוב...

  📂 קבצי פלט:
  📄 shulchan_aruch_v2.jsonl  (1,498,758 bytes)
  📄 shulchan_aruch_v2_full.json  (1,575,218 bytes)
  📄 validation_report

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ shulchan_aruch_v2.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ shulchan_aruch_v2_full.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ validation_report.txt

✅ כל הקבצים הורדו!
